In [1]:
import re
import osmium
import subprocess
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from geopy.geocoders import Nominatim

In [2]:
os.getcwd()
os.chdir(os.path.join(os.getcwd(), 'backend'))

In [ ]:
LEVEL_HEIGHT = 3.4

# https://wiki.openstreetmap.org/wiki/Simple_3D_buildings#Other_roof_tags


def _feet_to_meters(s):
    r = re.compile(r"([0-9]*\.?[0-9]+)'([0-9]*\.?[0-9]+)?\"?")
    m = r.findall(s)[0]
    if len(m[0]) > 0 and len(m[1]) > 0:
        m = float(m[0]) + float(m[1]) / 12.0
    elif len(m[0]) > 0:
        m = float(m[0])
    return m * 0.3048


def _get_height(tags):
    if 'height' in tags:
        # already accounts for roof
        if '\'' in tags['height'] or '\"' in tags['height']:
            return _feet_to_meters(tags['height'])
        r = re.compile(r"[-+]?\d*\.\d+|\d+")
        return float(r.findall(tags['height'])[0])
    if 'levels' in tags:
        roof_height = 0
        if 'roof_height' in tags:
            if '\'' in tags['roof_height'] or '\"' in tags['roof_height']:
                roof_height = _feet_to_meters(tags['roof_height'])
            else:
                r = re.compile(r"[-+]?\d*\.\d+|\d+")
                roof_height = float(r.findall(tags['roof_height'])[0])

        # does not account for roof height
        height = float(tags['levels']) * LEVEL_HEIGHT
        if 'roof_levels' in tags and roof_height == 0:
            height += float(tags['roof_levels']) * LEVEL_HEIGHT
        return height
    return 7.0


def _get_min_height(tags):
    if 'min_height' in tags:
        # already accounts for roof
        if '\'' in tags['min_height'] or '\"' in tags['min_height']:
            return _feet_to_meters(tags['min_height'])
        r = re.compile(r"[-+]?\d*\.\d+|\d+")
        return float(r.findall(tags['min_height'])[0])
    if 'min_level' in tags:
        height = float(tags['min_level']) * LEVEL_HEIGHT
        return height
    return 0.0


class BuildingHandler(osmium.SimpleHandler):

    def __init__(self):
        osmium.SimpleHandler.__init__(self)
        self.geometry = []       # WKB bytes
        self.height = []
        self.min_height = []
        self.osm_id = []         # numeric id
        self.osm_type = []       # 'W' or 'R'
        self.wkbfab = osmium.geom.WKBFactory()

    def get_gdf(self):
        geom = gpd.GeoSeries.from_wkb(self.geometry, crs='EPSG:4326')
        gdf = gpd.GeoDataFrame({
            'osm_id': self.osm_id,
            'osm_type': self.osm_type,
            'min_height': pd.Series(self.min_height, dtype='float'),
            'height': pd.Series(self.height, dtype='float'),
            'geometry': geom
        }, index=geom.index)
        return gdf

    def area(self, a):
        id = int(a.orig_id())
        osm_type = 'W' if a.from_way() else 'R'

        tags = a.tags
        # Qualifiers
        if not ('building' in tags or 'building:part' in tags or tags.get('type', None) == 'building'):
            return
        # Disqualifiers
        if (tags.get('location', None) == 'underground' or 'bridge' in tags):
            return
        try:
            poly = self.wkbfab.create_multipolygon(a)
            height = _get_height(tags)
            min_height = _get_min_height(tags)

            self.geometry.append(poly)
            self.height.append(height)
            self.min_height.append(min_height)
            self.osm_id.append(id)
            self.osm_type.append(osm_type)
            
        except Exception as e:
            print(e)
            print(a)

def save_buildings_geojson(handler: BuildingHandler, out_path: str) -> gpd.GeoDataFrame:
    """
    Build a GeoDataFrame (with osm_id, osm_type, height, min_height, geometry)
    and save it as GeoJSON. Returns the GeoDataFrame.
    """
    gdf = handler.get_gdf()
    gdf.to_file(out_path, driver="GeoJSON")
    return

In [13]:
city_full = 'Chicago, Illinois, USA'
city = 'chi'
filename = 'data/%s.osm.pbf' % (city)
input_filename = 'data/north-america-latest.osm.pbf'

geolocator = Nominatim(user_agent='uic')
location = geolocator.geocode(city_full).raw
 
input_filename = 'data/north-america-latest.osm.pbf'

south, north, west, east = map(float, location['boundingbox'])
bbox = f"{west},{south},{east},{north}"

In [ ]:
# apt get install osmium-tool

subprocess.run([
    "osmium", "extract",
    "-b", bbox,
    "-o", filename,
    "--overwrite",
    input_filename
])



CompletedProcess(args=['osmium', 'extract', '-b', '-87.9400876,41.644531,-87.5240812,42.0230396', '-o', 'data/chi.osm.pbf', '--overwrite', 'data/north-america-latest.osm.pbf'], returncode=0)

In [27]:
h = BuildingHandler()
h.apply_file(filename, locations=True)

invalid area (area_id=324778144)
a324778144: num_rings=(0, 0), tags={addr:city=Chicago,addr:housenumber=2150,addr:po...}


In [ ]:
# gdf = h.get_gdf()
gdf.to_feather('data/%s_buildings.feather' % city, compression='lz4')


In [38]:
gdf = gpd.read_feather('data/chi_buildings.feather')

In [58]:
xmin, ymin, xmax, ymax = -87.655, 41.865, -87.605, 41.895
gdf_cut = gdf.cx[xmin:xmax, ymin:ymax]

In [59]:
gdf_cut.to_file('./data/chi_downtown_buildings.geojson', driver="GeoJSON")

In [28]:
import osmnx as ox
from pyrosm import OSM
# path to your local PBF
pbf_path = "./data/chi.osm.pbf"

osm = OSM(pbf_path)

nodes, edges = osm.get_network(
    network_type="driving",
    nodes=True,
    extra_attributes=["name", "highway", "maxspeed"]
)

# G = osm.to_graph(nodes, edges, graph_type="networkx")  # already EPSG:4326

# G = ox.simplify_graph(G)
# G = ox.project_graph(G)

In [16]:
edges.columns

Index(['access', 'area', 'bicycle', 'bicycle_road', 'bridge', 'busway',
       'cycleway', 'foot', 'footway', 'highway', 'junction', 'lanes', 'lit',
       'maxspeed', 'motorcar', 'motorroad', 'motor_vehicle', 'name', 'oneway',
       'psv', 'ref', 'service', 'segregated', 'sidewalk', 'smoothness',
       'surface', 'tracktype', 'tunnel', 'turn', 'width', 'id', 'timestamp',
       'version', 'tags', 'osm_type', 'geometry', 'u', 'v', 'length'],
      dtype='object')

In [29]:
edges = edges[['id', 'osm_type', 'geometry', 'width', 'length']]

In [30]:
edges = edges.set_crs(4326)             # ensure WGS84 (lon/lat)

In [ ]:
edges.to_feather("./data/chi_roads.feather")

In [33]:
xmin, ymin, xmax, ymax = -87.655, 41.865, -87.605, 41.895
edges = edges.cx[xmin:xmax, ymin:ymax]

In [ ]:
edges.to_file("./data/chi_downtown_edges.geojson", driver="GeoJSON")